# Notebook Databricks — Transformação Silver

**Objetivo:** ler a tabela `lakehouse_catalog.bronze.livros_raw`, aplicar limpeza, tipagem e deduplicação, e gravar o resultado em `lakehouse_catalog.silver.livros`.

Pré-requisitos:
- Executar `sql/ddl/02_create_silver_tables.sql`.
- Executar o notebook `02_ingest_bronze.ipynb`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

from common.spark_utils import get_spark_session
from common.silver_utils import (
    clean_preco,
    clean_rating,
    clean_disponibilidade,
    build_id_livro,
    deduplicate_latest,
)
from pyspark.sql import functions as F

spark = get_spark_session()

In [ ]:
df_bronze = spark.table("lakehouse_catalog.bronze.livros_raw")
df_bronze.count()

In [ ]:
df_silver = (
    df_bronze
    .pipe(clean_preco)
    .pipe(clean_rating)
    .pipe(clean_disponibilidade)
    .pipe(build_id_livro)
    .withColumn("dt_coleta", F.to_date("dt_coleta"))
    .withColumn("dt_atualizacao", F.current_timestamp())
    .select(
        "id_livro", "titulo", "categoria", "preco",
        "rating", "disponivel", "dt_coleta", "dt_atualizacao",
    )
)

df_silver = deduplicate_latest(df_silver, ["id_livro"], "dt_atualizacao")
df_silver.show(5, truncate=False)

In [ ]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("lakehouse_catalog.silver.livros")

print(f"Registros gravados em lakehouse_catalog.silver.livros: {df_silver.count()}")